# Lichess → Nagato NNUE Training Pipeline

Stream Lichess `.pgn.zst` files, filter for quality games, sample positions,
evaluate with Stockfish, and output Nagato-compatible training data.

**Filters:**
- Both players rated ≥ 2000
- Time control ≥ 3+0 (no bullet/ultrabullet)
- Normal termination only (no abandons, timeouts, rule violations)
- Skip first 8 moves (opening book territory)
- 1–3 random positions per game

**Output:** Binary file matching Nagato trainer format — one entry per position:
```
FEN string (null-terminated) | i16 cp_score | u8 WDL (2=win, 1=draw, 0=loss)
```

In [ ]:
import chess
import chess.pgn
import chess.engine
import zstandard
import io
import os
import struct
import random
import time
from pathlib import Path

## Configuration

In [ ]:
GAMES_DIR = Path("games")
OUTPUT_FILE = "lichess_training_data.bin"
STOCKFISH_PATH = "/opt/homebrew/bin/stockfish"
SF_DEPTH = 14
SF_THREADS = 4
SF_HASH_MB = 256

MIN_ELO = 2000
MIN_TIME_BASE = 180  # seconds (3 minutes)
SKIP_PLIES = 16      # skip first 8 full moves
POSITIONS_PER_GAME = 2
MAX_GAMES = None      # set to int to limit, None for all
PROGRESS_EVERY = 500  # print stats every N games

print(f"Stockfish: {STOCKFISH_PATH}")
print(f"Games dir: {GAMES_DIR}")
print(f"ZST files: {sorted(GAMES_DIR.glob('*.pgn.zst'))}")

## Filtering & Parsing Helpers

In [ ]:
def parse_time_control(tc_str):
    """Parse Lichess TimeControl header. Returns (base_seconds, increment)."""
    if not tc_str or tc_str == "-":
        return (0, 0)
    parts = tc_str.split("+")
    try:
        base = int(parts[0])
        inc = int(parts[1]) if len(parts) > 1 else 0
        return (base, inc)
    except ValueError:
        return (0, 0)

def passes_filter(headers):
    """Check if game headers pass our quality filters."""
    # Rating filter
    try:
        w_elo = int(headers.get("WhiteElo", "0"))
        b_elo = int(headers.get("BlackElo", "0"))
    except ValueError:
        return False
    if w_elo < MIN_ELO or b_elo < MIN_ELO:
        return False

    # Time control filter — no bullet
    base, _ = parse_time_control(headers.get("TimeControl", ""))
    if base < MIN_TIME_BASE:
        return False

    # Termination filter
    term = headers.get("Termination", "")
    if term != "Normal":
        return False

    # Must have a decisive or drawn result
    result = headers.get("Result", "*")
    if result not in ("1-0", "0-1", "1/2-1/2"):
        return False

    return True

def result_to_wdl(result_str, side_to_move):
    """Convert PGN result to WDL u8 from STM perspective. 2=win, 1=draw, 0=loss."""
    if result_str == "1/2-1/2":
        return 1
    white_wins = result_str == "1-0"
    stm_is_white = side_to_move == chess.WHITE
    if white_wins == stm_is_white:
        return 2  # STM won
    return 0      # STM lost

# Quick sanity test
assert parse_time_control("180+2") == (180, 2)
assert parse_time_control("600+0") == (600, 0)
assert result_to_wdl("1-0", chess.WHITE) == 2
assert result_to_wdl("1-0", chess.BLACK) == 0
assert result_to_wdl("1/2-1/2", chess.WHITE) == 1
print("Helpers OK")

## ZST Streaming Reader

In [ ]:
def stream_pgn_games(zst_path):
    """Yield chess.pgn.Game objects from a .pgn.zst file, streaming."""
    dctx = zstandard.ZstdDecompressor()
    with open(zst_path, "rb") as fh:
        reader = dctx.stream_reader(fh)
        text_stream = io.TextIOWrapper(reader, encoding="utf-8", errors="replace")
        while True:
            game = chess.pgn.read_game(text_stream)
            if game is None:
                break
            yield game

# Quick test — read 3 games
zst_files = sorted(GAMES_DIR.glob("*.pgn.zst"))
count = 0
for game in stream_pgn_games(zst_files[0]):
    w = game.headers.get("WhiteElo", "?")
    b = game.headers.get("BlackElo", "?")
    tc = game.headers.get("TimeControl", "?")
    print(f"Game {count+1}: {w} vs {b}, TC={tc}, Result={game.headers.get('Result')}")
    count += 1
    if count >= 3:
        break
print(f"Streaming works — read {count} games")

## Position Sampler + Stockfish Evaluator

In [ ]:
def sample_positions(game, n=POSITIONS_PER_GAME, skip_plies=SKIP_PLIES):
    """Sample n random positions from a game, skipping the opening.
    Returns list of (ply, board_copy) tuples."""
    board = game.board()
    positions = []
    ply = 0
    for move in game.mainline_moves():
        board.push(move)
        ply += 1
        if ply > skip_plies and not board.is_game_over():
            positions.append((ply, board.copy()))

    if not positions:
        return []

    k = min(n, len(positions))
    return random.sample(positions, k)

# Test sampler
for game in stream_pgn_games(zst_files[0]):
    if passes_filter(game.headers):
        samples = sample_positions(game)
        for ply, board in samples:
            print(f"  ply={ply} turn={'W' if board.turn else 'B'} fen={board.fen()[:50]}...")
        break
print("Sampler OK")

## Binary Writer (Nagato 40-byte Packed Format)

Each entry is exactly 40 bytes, matching `datagen::write_entry` / `trainer::parse_entry`:
- `[0..32]` packed board (nibble per square, 2 per byte)
- `[32]` side to move (0=White, 1=Black)
- `[33]` castling (K=1, Q=2, k=4, q=8)
- `[34]` en passant file (0-7, 255=none)
- `[35]` padding
- `[36..38]` score i16 LE (from White's perspective)
- `[38]` WDL (0/1/2 from side-to-move perspective)
- `[39]` padding

In [ ]:
ENTRY_SIZE = 40

# Piece encoding matching Rust datagen::pack_board()
PIECE_NIBBLE = {
    (chess.PAWN,   chess.WHITE): 1,  (chess.KNIGHT, chess.WHITE): 2,
    (chess.BISHOP, chess.WHITE): 3,  (chess.ROOK,   chess.WHITE): 4,
    (chess.QUEEN,  chess.WHITE): 5,  (chess.KING,   chess.WHITE): 6,
    (chess.PAWN,   chess.BLACK): 7,  (chess.KNIGHT, chess.BLACK): 8,
    (chess.BISHOP, chess.BLACK): 9,  (chess.ROOK,   chess.BLACK): 10,
    (chess.QUEEN,  chess.BLACK): 11, (chess.KING,   chess.BLACK): 12,
}

def pack_board(board):
    """Pack board into 32 bytes matching Rust nibble format."""
    packed = bytearray(32)
    for sq in range(64):
        piece = board.piece_at(sq)
        if piece is None:
            continue
        nibble = PIECE_NIBBLE[(piece.piece_type, piece.color)]
        byte_idx = sq // 2
        if sq % 2 == 0:
            packed[byte_idx] |= nibble
        else:
            packed[byte_idx] |= (nibble << 4)
    return bytes(packed)

def encode_castling(board):
    rights = 0
    if board.has_kingside_castling_rights(chess.WHITE):  rights |= 1
    if board.has_queenside_castling_rights(chess.WHITE): rights |= 2
    if board.has_kingside_castling_rights(chess.BLACK):  rights |= 4
    if board.has_queenside_castling_rights(chess.BLACK): rights |= 8
    return rights

def write_entry(f, board, score_white, wdl):
    """Write one 40-byte training entry in Nagato packed format."""
    packed = pack_board(board)
    entry = bytearray(ENTRY_SIZE)
    entry[0:32] = packed
    entry[32] = 0 if board.turn == chess.WHITE else 1
    entry[33] = encode_castling(board)
    entry[34] = chess.square_file(board.ep_square) if board.ep_square is not None else 255
    entry[35] = 0
    struct.pack_into("<h", entry, 36, max(-32000, min(32000, score_white)))
    entry[38] = wdl
    entry[39] = 0
    f.write(entry)

def read_entry(f):
    """Read one 40-byte entry back (for verification)."""
    data = f.read(ENTRY_SIZE)
    if len(data) < ENTRY_SIZE:
        return None
    side = "w" if data[32] == 0 else "b"
    score = struct.unpack("<h", data[36:38])[0]
    wdl = data[38]
    # Reconstruct board for display
    board = chess.Board(fen=None)
    board.clear()
    for sq in range(64):
        byte_idx = sq // 2
        nibble = (data[byte_idx] & 0x0F) if sq % 2 == 0 else ((data[byte_idx] >> 4) & 0x0F)
        if nibble == 0:
            continue
        for (pt, color), val in PIECE_NIBBLE.items():
            if val == nibble:
                board.set_piece_at(sq, chess.Piece(pt, color))
                break
    return (board.board_fen(), score, wdl, side)

# Test roundtrip
import tempfile
test_board = chess.Board("rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq e3 0 1")
with tempfile.NamedTemporaryFile() as tmp:
    write_entry(tmp, test_board, 35, 2)
    tmp.seek(0)
    e = read_entry(tmp)
    print(f"Entry: score={e[1]} wdl={e[2]} stm={e[3]} fen={e[0]}")
print("Writer OK (40-byte packed format)")

## Main Pipeline

Stream games → filter → sample positions → Stockfish eval → write binary.

In [ ]:
def run_pipeline(zst_files, output_path, max_games=MAX_GAMES):
    stats = {
        "games_seen": 0, "games_passed": 0, "games_skipped": 0,
        "positions_sampled": 0, "positions_evaluated": 0,
        "sf_errors": 0,
    }
    t0 = time.time()

    engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)
    engine.configure({"Threads": SF_THREADS, "Hash": SF_HASH_MB})

    with open(output_path, "wb") as out:
        for zst_path in zst_files:
            print(f"\n--- Processing {zst_path.name} ---")
            for game in stream_pgn_games(zst_path):
                stats["games_seen"] += 1

                if not passes_filter(game.headers):
                    stats["games_skipped"] += 1
                else:
                    stats["games_passed"] += 1
                    result_str = game.headers["Result"]

                    sampled = sample_positions(game)
                    stats["positions_sampled"] += len(sampled)

                    for ply, board in sampled:
                        try:
                            info = engine.analyse(board, chess.engine.Limit(depth=SF_DEPTH))
                            score = info["score"].pov(chess.WHITE)
                            if score.is_mate():
                                cp_white = 30000 if score.mate() > 0 else -30000
                            else:
                                cp_white = score.score()
                            wdl = result_to_wdl(result_str, board.turn)
                            write_entry(out, board, cp_white, wdl)
                            stats["positions_evaluated"] += 1
                        except Exception as e:
                            stats["sf_errors"] += 1

                if stats["games_seen"] % PROGRESS_EVERY == 0:
                    elapsed = time.time() - t0
                    gps = stats["games_seen"] / elapsed if elapsed > 0 else 0
                    print(f"  [{elapsed:.0f}s] seen={stats['games_seen']:,} passed={stats['games_passed']:,} "
                          f"pos={stats['positions_evaluated']:,} ({gps:.1f} games/s)")

                if max_games and stats["games_passed"] >= max_games:
                    break
            if max_games and stats["games_passed"] >= max_games:
                break

    engine.quit()
    elapsed = time.time() - t0
    file_size = os.path.getsize(output_path)
    print(f"\n=== Pipeline complete ===")
    print(f"Time: {elapsed:.1f}s")
    print(f"Games seen: {stats['games_seen']:,}")
    print(f"Games passed filter: {stats['games_passed']:,} ({100*stats['games_passed']/max(1,stats['games_seen']):.1f}%)")
    print(f"Positions evaluated: {stats['positions_evaluated']:,}")
    print(f"SF errors: {stats['sf_errors']}")
    print(f"Output: {output_path} ({file_size:,} bytes, {file_size//ENTRY_SIZE} entries)")
    return stats

## Run — Small Test (1000 quality games)

In [ ]:
zst_files = sorted(GAMES_DIR.glob("*.pgn.zst"))
print(f"Found {len(zst_files)} ZST files")

# Small test run — 1000 games that pass filter
stats = run_pipeline(zst_files, OUTPUT_FILE, max_games=1000)

## Verify Output

In [ ]:
# Read back and inspect entries
file_size = os.path.getsize(OUTPUT_FILE)
n_entries = file_size // ENTRY_SIZE

with open(OUTPUT_FILE, "rb") as f:
    entries = []
    for _ in range(min(10, n_entries)):
        e = read_entry(f)
        if e is None:
            break
        entries.append(e)

import pandas as pd
df = pd.DataFrame(entries, columns=["fen", "score_white", "wdl", "stm"])
print(f"First {len(df)} entries:")
display(df)

print(f"\nTotal entries: {n_entries:,}")
print(f"File size: {file_size:,} bytes ({file_size/1e6:.2f} MB)")
print(f"Entry size: {ENTRY_SIZE} bytes")

## Full Run

For production runs, use `pipeline_parallel.py` (8 workers, depth 10, ~55 pos/s).
For notebook exploration, adjust `MAX_GAMES` above and run the test cell.

Throughput notes:
- Depth 10: ~227 pos/s per SF thread → ~55 pos/s with parallel pipeline
- Depth 14: ~10 pos/s per SF thread (6x slower, marginal quality gain)
- 1M positions ≈ 5 hours with parallel pipeline at depth 10

In [ ]:
# Uncomment to run on full dataset:
# stats = run_pipeline(zst_files, "lichess_training_large.bin", max_games=None)